## Import Library and Packages

In [103]:
pip install pandas numpy scikit-learn xgboost sentence-transformers nltk --quite


Usage:   
  pip3 install [options] <requirement specifier> [package-index-options] ...
  pip3 install [options] -r <requirements file> [package-index-options] ...
  pip3 install [options] [-e] <vcs project url> ...
  pip3 install [options] [-e] <local project path> ...
  pip3 install [options] <archive url/path> ...

no such option: --quite


In [104]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, roc_auc_score
from xgboost import XGBClassifier
from sentence_transformers import SentenceTransformer

from typing import Dict, List

plt.style.use("seaborn-v0_8")

## Import Dataset

In [105]:
df = pd.read_csv("new_confident_dataset.csv")

In [106]:
df["confidence_level"] = df["confidence_level"].str.strip().str.lower()


In [107]:
df["confidence_label"] = df["confidence_level"].map({
    "low": 0,
    "medium": 0,  # Treating medium as low confidence
    "high": 1
})


In [108]:
df = df.dropna(subset=["confidence_label"])
df["confidence_label"] = df["confidence_label"].astype(int)

In [109]:
print("Final class distribution:")
print(df["confidence_label"].value_counts())

Final class distribution:
confidence_label
1    6777
0    3262
Name: count, dtype: int64


In [110]:
def clean_text(text):
    text = str(text).lower().strip()
    text = re.sub(r"\s+", " ", text)
    return text

df["clean_text"] = df["transcription"].apply(clean_text)

## Feature Extraction

In [111]:
HEDGE_WORDS = {
    "maybe", "perhaps", "probably", "possibly",
    "i think", "i guess", "i suppose", "i believe",
    "not sure", "kind of", "sort of", "somewhat",
    "might", "could", "would", "should"
}

CERTAINTY_WORDS = {
    "definitely", "sure", "certainly", "i know",
    "absolutely", "clearly", "obviously",
    "without doubt", "for sure", "guarantee",
    "ensure", "confirmed", "will"
}

FILLERS = {
    "uh", "um", "er", "ah",
    "like", "you know", "i mean", "well"
}

MODAL_WORDS = {"might", "could", "should", "would", "may"}

NEGATIONS = {"not", "no", "never", "don't", "didn't", "can't", "won't"}

OWNERSHIP_WORDS = {
    "i will", "i'll", "i can",
    "let me", "i'll handle",
    "i'll resolve", "i'll fix"
}

ESCALATION_WORDS = {
    "supervisor", "manager",
    "another department", "escalate",
    "transfer you"
}

DELAY_WORDS = {
    "call back", "get back to you",
    "follow up", "check on",
    "look into", "in progress"
}

def count_phrases(text, phrase_set):
    return sum(text.count(p) for p in phrase_set)

def extract_linguistic_features(text):
    text = text.lower()
    words = re.findall(r"\w+", text)
    word_count = len(words) if words else 1

    hedge_count = count_phrases(text, HEDGE_WORDS)
    certainty_count = count_phrases(text, CERTAINTY_WORDS)
    filler_count = count_phrases(text, FILLERS)
    ownership_count = count_phrases(text, OWNERSHIP_WORDS)
    escalation_count = count_phrases(text, ESCALATION_WORDS)
    delay_count = count_phrases(text, DELAY_WORDS)

    negation_count = sum(1 for w in words if w in NEGATIONS)
    modal_count = sum(1 for w in words if w in MODAL_WORDS)

    question_count = text.count("?")
    exclamation_count = text.count("!")

    first_person_ratio = words.count("i") / word_count

    avg_word_length = np.mean([len(w) for w in words]) if words else 0

    # Behavioral scores
    hesitation_score = (hedge_count + filler_count + modal_count) / word_count
    decisiveness_score = certainty_count / word_count
    ownership_score = ownership_count / word_count
    escalation_ratio = escalation_count / word_count
    delay_ratio = delay_count / word_count

    # Composite confidence index
    confidence_index = (
        2.0 * ownership_score +
        1.5 * decisiveness_score -
        2.0 * hesitation_score -
        1.2 * escalation_ratio -
        1.0 * delay_ratio -
        1.0 * (negation_count / word_count)
    )

    features = {
        "hedge_count": hedge_count,
        "certainty_count": certainty_count,
        "filler_count": filler_count,
        "ownership_count": ownership_count,
        "escalation_count": escalation_count,
        "delay_count": delay_count,
        "negation_count": negation_count,
        "modal_count": modal_count,
        "question_count": question_count,
        "exclamation_count": exclamation_count,
        "first_person_ratio": first_person_ratio,
        "avg_word_length": avg_word_length,
        "sentence_length": word_count,
        "hesitation_score": hesitation_score,
        "decisiveness_score": decisiveness_score,
        "ownership_score": ownership_score,
        "confidence_index": confidence_index
    }

    return features

In [112]:
class ConfidenceFeatureExtractor:
    """Enhanced version with comprehensive linguistic markers"""
    
    def __init__(self):
        # HEDGING WORDS - 60+ markers
        self.hedging_words = [
            # Basic hedges
            'maybe', 'perhaps', 'probably', 'possibly', 'potentially',
            'might', 'may', 'could', 'would', 'should',
            # Cognitive hedges
            'i think', 'i believe', 'i feel', 'i guess', 'i suppose',
            'i imagine', 'i assume', 'i reckon', 'i suspect',
            # Degree hedges
            'kind of', 'sort of', 'somewhat', 'rather', 'fairly',
            'quite', 'relatively', 'comparatively', 'reasonably',
            'pretty much', 'more or less', 'to some extent',
            # Approximators
            'approximately', 'roughly', 'about', 'around', 'nearly',
            'almost', 'practically', 'virtually', 'essentially',
            # Probability hedges
            'likely', 'unlikely', 'apt to', 'liable to', 'tends to',
            'seems to', 'appears to', 'looks like', 'sounds like',
            # Conditional hedges
            'if possible', 'if available', 'if applicable', 'depending on',
            'subject to', 'provided that', 'assuming that',
            # Minimizers
            'a bit', 'a little', 'slightly', 'marginally',
            'to a degree', 'in a way', 'in a sense',
            # Tentative language
            'tend to', 'seem to', 'appear to', 'suggest that',
            'indicate that', 'imply that',
            # Epistemic hedges
            'as far as i know', 'to my knowledge', 'in my opinion',
            'in my view', 'from my perspective', 'it seems that',
            'it appears that', 'i would say',
            # Additional hedges
            'arguably', 'presumably', 'conceivably', 'theoretically',
            'technically', 'supposedly', 'ostensibly',
            'it could be that', 'it may be that', 'it might be that',
            'one could say', 'it is possible that',
            'more or less', 'give or take',
            'in general', 'in most cases', 'in some cases',
            'at times', 'from time to time',
            'it tends to be', 'it often seems',
        ]
        
        # FILLER WORDS - 50+ markers
        self.filler_words = [
            # Classic fillers
            'um', 'uh', 'er', 'ah', 'eh', 'mm', 'hmm', 'uhh', 'umm',
            # Discourse markers
            'like', 'you know', 'i mean', 'you see',
            'you know what i mean', 'know what i\'m saying',
            # Temporal fillers
            'well', 'so', 'now', 'then', 'anyway', 'anyways',
            'alright', 'okay', 'ok',
            # Intensifiers used as fillers
            'actually', 'basically', 'literally', 'seriously',
            'honestly', 'really', 'totally', 'completely',
            # Repair markers
            'i mean to say', 'what i mean is', 'let me think',
            'how do i put it', 'how should i say',
            # Stalling phrases
            'let me see', 'let me check', 'give me a second',
            'just a moment', 'hold on', 'wait a minute',
            'one second', 'bear with me',
            # Continuation markers
            'and everything', 'and stuff', 'and all that',
            'or something', 'or whatever', 'and so on',
            'et cetera', 'and the like',
            # More fillers
            'right', 'okay then', 'so yeah', 'yeah',
            'well then', 'you know like', 'kind of like',
            'basically speaking', 'to be honest',
            'frankly', 'at the end of the day',
            'by the way', 'on that note',
            'if you will', 'sort of like',
            'just saying', 'that being said',
        ]
        
        # CONFIDENCE WORDS - 70+ markers
        self.confidence_words = [
            # Certainty markers
            'definitely', 'certainly', 'absolutely', 'undoubtedly',
            'without doubt', 'no doubt', 'for sure', 'for certain',
            # Knowledge indicators
            'know', 'knows', 'knew', 'knowing', 'understand',
            'clear', 'clearly', 'obvious', 'obviously', 'evident',
            'evidently', 'apparent', 'apparently',
            # Assurance words
            'sure', 'positive', 'confident', 'certain', 'convinced',
            'assured', 'secure', 'firm',
            # Future commitment
            'will', 'shall', 'going to', 'guarantee', 'guaranteed',
            'promise', 'promised',
            # Strong affirmatives
            'yes', 'correct', 'right', 'exactly', 'precisely',
            'indeed', 'truly', 'surely',
            # Expertise markers
            'expert', 'experienced', 'qualified', 'trained',
            'specialized', 'professional', 'competent',
            # Factual language
            'fact', 'facts', 'proven', 'verified', 'confirmed',
            'established', 'demonstrated', 'shown',
            # Direct statements
            'is', 'are', 'was', 'were', 'been',
            # Solution-oriented
            'solution', 'solve', 'fix', 'resolve', 'handle',
            'address', 'deal with', 'take care of',
            # Immediacy
            'now', 'immediately', 'right away', 'right now',
            'instantly', 'straightaway', 'at once',
            # Strong certainty words
            'unquestionably', 'indisputably', 'irrefutable',
            'categorically', 'decisively', 'conclusively',
            'assuredly', 'unmistakably',
            'validated', 'substantiated',
            'guarantees', 'ensures',
            'commit', 'committed',
            'resolved', 'accomplished',
            'definitive', 'concrete',
            'authoritative', 'credible',
            'dependable', 'reliable',
        ]
        
        # UNCERTAINTY PHRASES - 30+ markers
        self.uncertainty_phrases = [
            # Direct uncertainty
            "i don't know", "i dunno", "i'm not sure", "not sure",
            "i'm uncertain", "i'm unsure", "i have no idea",
            # Confusion
            "i'm confused", "that's confusing", "unclear",
            "i don't understand", "i'm lost",
            # Lack of information
            "i don't have", "i lack", "i'm missing",
            "can't tell", "hard to say", "difficult to say",
            # Inability
            "i can't", "i cannot", "unable to", "not able to",
            "don't know how",
            # Doubt
            "i doubt", "doubtful", "questionable",
            # Need for help
            "need help", "need assistance", "need to check",
            "need to ask", "need to verify",
            # Deferral
            "ask someone else", "check with", "another department",
            # More uncertainty expressions
            "i'm not certain",
            "i'm not entirely sure",
            "i can't confirm",
            "i can't verify",
            "not entirely clear",
            "it depends",
            "it varies",
            "uncertain at this point",
            "still checking",
            "pending confirmation",
            "awaiting information",
            "i'm guessing",
            "i'm speculating",
        ]
        
        # ASSERTIVE WORDS - 25+ markers
        self.assertive_words = [
            'must', 'have to', 'need to', 'got to', 'gotta',
            'ought to', 'should', 'shall', 'will',
            'require', 'required', 'requires', 'necessary',
            'essential', 'crucial', 'critical', 'vital',
            'important', 'imperative', 'mandatory',
            'ensure', 'make sure', 'guarantee',
            # Strong directives
            'cannot', 'must not',
            'obligated', 'compulsory',
            'non-negotiable', 'enforced',
            'strictly', 'demand',
            'insist', 'direct',
            'authorize', 'approved',
            'enforce', 'implement',
        ]
        
        # POLITENESS MARKERS - 20+ markers
        self.politeness_markers = [
            'sorry', 'apologize', 'apologies', 'excuse me',
            'pardon', 'please', 'kindly', 'thank you',
            'thanks', 'appreciate', 'grateful',
            'would you mind', 'could you please', 'if possible',
            'with respect', 'respectfully', 'if i may',
            # Additional politeness phrases
            'much appreciated',
            'thank you very much',
            'i appreciate your patience',
            'i appreciate your understanding',
            'my pleasure',
            'happy to help',
            'feel free',
            'at your convenience',
            'whenever you’re ready',
        ]
        
        # ABSOLUTE WORDS - 30+ markers
        self.absolute_words = [
            'all', 'every', 'everyone', 'everybody', 'everything',
            'always', 'forever', 'constantly', 'continually',
            'none', 'nothing', 'nobody', 'no one', 'nowhere',
            'never', 'complete', 'completely', 'total', 'totally',
            'entire', 'entirely', 'whole', 'wholly',
            'full', 'fully', 'absolute', 'absolutely',
            # Strong absolutes
            'definitively',
            'comprehensively',
            'universally',
            'unconditionally',
            'irrevocably',
            'permanently',
            'entirety',
            'in every case',
            'without exception',
        ]
        
        # QUESTION WORDS - 15+ markers
        self.question_words = [
            'what', 'when', 'where', 'why', 'who', 'whom',
            'which', 'whose', 'how',
            'is it', 'are there', 'do you', 'can you',
            'could you', 'would you',
            # More interrogatives
            'does it',
            'did you',
            'will you',
            'shall we',
            'why is',
            'how come',
            'what if',
            'who else',
        ]
        
        # POSITIVE EMOTIONS - 13 markers
        self.positive_emotion_words = [
            'happy', 'glad', 'pleased', 'delighted', 'excited',
            'confident', 'proud', 'satisfied', 'content',
            'great', 'excellent', 'perfect', 'wonderful',
            # More positive emotion terms
            'thrilled', 'joyful', 'optimistic',
            'relieved', 'grateful', 'encouraged',
            'motivated', 'inspired',
            'hopeful', 'enthusiastic',
            'fantastic', 'amazing',
        ]
        
        # NEGATIVE EMOTIONS - 12 markers
        self.negative_emotion_words = [
            'worried', 'concerned', 'anxious', 'nervous',
            'hesitant', 'reluctant', 'uncomfortable', 'uneasy',
            'stressed', 'frustrated', 'confused', 'unsure',
            # More negative emotion terms
            'upset', 'disappointed',
            'overwhelmed', 'irritated',
            'annoyed', 'discouraged',
            'hopeless', 'dissatisfied',
            'regretful', 'angry',
        ]
        
        # ESCALATION PHRASES - 9 markers
        self.escalation_phrases = [
            'transfer you', 'transfer to', 'supervisor',
            'manager', 'specialist', 'someone who can',
            'better suited', 'another department', 'escalate',
            # More escalation markers
            'let me transfer',
            'connect you with',
            'pass this to',
            'refer you to',
            'forward this to',
            'escalate this matter',
            'bring this to the attention of',
        ]
        
        # OWNERSHIP PHRASES - 10 markers
        self.ownership_phrases = [
            'i will', 'i can', "i'll handle", "i'll take care",
            'let me help', 'i can help', "i'll assist",
            "i'll resolve", "i'll fix", "i'll solve",
            # Strong ownership language
            'i’ll personally',
            'i’ll make sure',
            'i’ll ensure',
            'i’ll follow through',
            'leave it with me',
            'i’ve got this',
            'i’ll handle this for you',
        ]
        
        # DELAY PHRASES - 9 markers
        self.delay_phrases = [
            'call back', 'get back to you', 'follow up',
            'check on that', 'look into', 'investigate',
            'find out', 'research', 'verify',
            # More delay markers
            'get back shortly',
            'revert back',
            'circle back',
            'update you',
            'keep you posted',
            'await response',
            'pending review',
            'in progress',
        ]
    
    def extract_features(self, text: str) -> Dict[str, float]:
        """Extract comprehensive confidence features"""
        text_lower = str(text).lower()
        words = text_lower.split()
        word_count = max(len(words), 1)
        
        features = {}
        
        # Basic ratios
        features['hedging_ratio'] = self._count_matches(text_lower, self.hedging_words) / word_count
        features['filler_ratio'] = self._count_matches(text_lower, self.filler_words) / word_count
        features['confidence_ratio'] = self._count_matches(text_lower, self.confidence_words) / word_count
        features['assertive_ratio'] = self._count_matches(text_lower, self.assertive_words) / word_count
        features['politeness_ratio'] = self._count_matches(text_lower, self.politeness_markers) / word_count
        features['absolute_ratio'] = self._count_matches(text_lower, self.absolute_words) / word_count
        
        # Phrase counts
        features['uncertainty_count'] = self._count_matches(text_lower, self.uncertainty_phrases)
        features['question_word_count'] = self._count_matches(text_lower, self.question_words)
        
        # Emotional markers
        features['positive_emotion_ratio'] = self._count_matches(text_lower, self.positive_emotion_words) / word_count
        features['negative_emotion_ratio'] = self._count_matches(text_lower, self.negative_emotion_words) / word_count
        
        # Call center specific
        features['escalation_count'] = self._count_matches(text_lower, self.escalation_phrases)
        features['ownership_count'] = self._count_matches(text_lower, self.ownership_phrases)
        features['delay_count'] = self._count_matches(text_lower, self.delay_phrases)
        
        # Punctuation
        features['question_ratio'] = text.count('?') / word_count
        features['exclamation_ratio'] = text.count('!') / word_count
        
        # Sentence structure
        features['avg_word_length'] = np.mean([len(w) for w in words]) if words else 0
        features['sentence_length'] = word_count
        
        # First person
        first_person = ['i ', 'me ', 'my ', 'mine ', 'myself ']
        features['first_person_ratio'] = sum(1 for fp in first_person if fp in text_lower + ' ') / word_count
        
        # Statement type
        features['is_statement'] = 1 if not text.strip().endswith('?') else 0
        
        # Composite score
        features['confidence_score'] = (
            features['confidence_ratio'] + 
            features['assertive_ratio'] +
            features['ownership_count'] / word_count -
            features['hedging_ratio'] -
            features['filler_ratio'] -
            features['uncertainty_count'] / word_count
        )
        
        return features
    
    def _count_matches(self, text: str, word_list: List[str]) -> int:
        """Count matches in text"""
        return sum(1 for item in word_list if item in text)
    
    def extract_batch(self, texts: List[str]) -> pd.DataFrame:
        """Extract features for multiple texts"""
        features_list = [self.extract_features(text) for text in texts]
        return pd.DataFrame(features_list)
    
    def get_feature_summary(self) -> pd.DataFrame:
        """Get summary of all markers"""
        summary_data = [
            {'Category': 'Hedging Words', 'Count': len(self.hedging_words)},
            {'Category': 'Filler Words', 'Count': len(self.filler_words)},
            {'Category': 'Confidence Words', 'Count': len(self.confidence_words)},
            {'Category': 'Uncertainty Phrases', 'Count': len(self.uncertainty_phrases)},
            {'Category': 'Assertive Words', 'Count': len(self.assertive_words)},
            {'Category': 'Politeness Markers', 'Count': len(self.politeness_markers)},
            {'Category': 'Absolute Words', 'Count': len(self.absolute_words)},
            {'Category': 'Question Words', 'Count': len(self.question_words)},
            {'Category': 'Positive Emotions', 'Count': len(self.positive_emotion_words)},
            {'Category': 'Negative Emotions', 'Count': len(self.negative_emotion_words)},
            {'Category': 'Escalation Phrases', 'Count': len(self.escalation_phrases)},
            {'Category': 'Ownership Phrases', 'Count': len(self.ownership_phrases)},
            {'Category': 'Delay Phrases', 'Count': len(self.delay_phrases)},
        ]
        return pd.DataFrame(summary_data)

In [113]:
feature_extractor = ConfidenceFeatureExtractor()
ling_df = feature_extractor.extract_batch(df['clean_text'].tolist())
# ling_feauters = df["clean_text"].apply(extract_linguistic_features)
# ling_df = pd.DataFrame(list(ling_feauters))

## NLP Model

In [114]:
embedder = SentenceTransformer("all-MiniLM-L6-v2")

embeddings = embedder.encode(
    df["clean_text"].tolist(),
    show_progress_bar=True,
    batch_size=32
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Batches:   0%|          | 0/314 [00:00<?, ?it/s]

In [115]:
print(f"Embeddings shape: {embeddings.shape}")

# Combine features
X = np.hstack([
    embeddings,
    ling_df.values
])

Embeddings shape: (10039, 384)


In [116]:
y = df["confidence_label"].values

print(f"\nFinal feature matrix shape: {X.shape}")
print(f"Target distribution: {np.bincount(y)}")


Final feature matrix shape: (10039, 404)
Target distribution: [3262 6777]


## Train, Test, Validation Dataset Split

In [117]:
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.3, stratify=y, random_state=42
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, stratify=y_temp, random_state=42
)

In [118]:
print(f"Train set: {X_train.shape}, class distribution: {np.bincount(y_train)}")
print(f"Val set: {X_val.shape}, class distribution: {np.bincount(y_val)}")
print(f"Test set: {X_test.shape}, class distribution: {np.bincount(y_test)}")

Train set: (7027, 404), class distribution: [2283 4744]
Val set: (1506, 404), class distribution: [ 490 1016]
Test set: (1506, 404), class distribution: [ 489 1017]


### Applying Scaler

In [119]:
scaler = StandardScaler()

num_ling_features = ling_df.shape[1]
X_train[:, -num_ling_features:] = scaler.fit_transform(X_train[:, -num_ling_features:])
X_val[:, -num_ling_features:] = scaler.transform(X_val[:, -num_ling_features:])
X_test[:, -num_ling_features:] = scaler.transform(X_test[:, -num_ling_features:])

## Logistic Regression Model

In [120]:
logreg = LogisticRegression(
    max_iter=1000,
    class_weight="balanced",
    random_state=42
)
logreg.fit(X_train, y_train)

LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42)

## XGB Model

In [121]:
xgb = XGBClassifier(
    n_estimators=300,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="binary:logistic",
    eval_metric="logloss",
    random_state=42
)

xgb.fit(X_train, y_train)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.8, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric='logloss',
              feature_types=None, feature_weights=None, gamma=None,
              grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.05, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=4, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=300, n_jobs=None,
              num_parallel_tree=None, ...)

In [122]:
def evaluate(model, X, y, dataset_name=""):
    print(f"\n{dataset_name} Results:")
    print("-" * 40)

    probs = model.predict_proba(X)[:, 1]
    preds = (probs >= 0.5).astype(int)

    print("\nConfusion Matrix:")
    print(confusion_matrix(y, preds))

    print("\nClassification Report:")
    print(classification_report(y, preds, target_names=["Low Confidence", "High Confidence"]))

    print(f"ROC-AUC Score: {roc_auc_score(y, probs):.4f}")
    print(f"Accuracy: {accuracy_score(y, preds):.4f}")


## Evaluation for Logistic Regression

In [123]:
evaluate(logreg, X_test, y_test, "Test Set")


Test Set Results:
----------------------------------------

Confusion Matrix:
[[429  60]
 [104 913]]

Classification Report:
                 precision    recall  f1-score   support

 Low Confidence       0.80      0.88      0.84       489
High Confidence       0.94      0.90      0.92      1017

       accuracy                           0.89      1506
      macro avg       0.87      0.89      0.88      1506
   weighted avg       0.90      0.89      0.89      1506

ROC-AUC Score: 0.9602
Accuracy: 0.8911


## Evaluation for XGB

In [124]:
evaluate(xgb, X_test, y_test, "Test Set")


Test Set Results:
----------------------------------------

Confusion Matrix:
[[397  92]
 [ 38 979]]

Classification Report:
                 precision    recall  f1-score   support

 Low Confidence       0.91      0.81      0.86       489
High Confidence       0.91      0.96      0.94      1017

       accuracy                           0.91      1506
      macro avg       0.91      0.89      0.90      1506
   weighted avg       0.91      0.91      0.91      1506

ROC-AUC Score: 0.9728
Accuracy: 0.9137


In [125]:
def predict_confidence(text, model, threshold=0.5):
    """Predict confidence level for a given text."""
    text = clean_text(text)

    # Generate embedding
    emb = embedder.encode([text])

    # Extract linguistic features
    extractor = ConfidenceFeatureExtractor()

    ling = extractor.extract_features(text)
    ling_vec = np.array(list(ling.values())).reshape(1, -1)
    ling_vec_scaled = scaler.transform(ling_vec)

    # Combine features
    X_input = np.hstack([emb, ling_vec_scaled])

    # Predict
    prob = model.predict_proba(X_input)[0][1]
    label = "high" if prob >= threshold else "low"

    return {
        "confidence_label": label,
        "confidence_score": float(prob),
        "linguistic_features": ling
    }


## Testing Samples

In [126]:
test_texts = [
    "I'm absolutely certain this is the right approach.",
    "Um, maybe we could try that, I guess?",
    "I don't know, perhaps it might work.",
    "This is definitely the best solution.",
    "Maybe i will try it after this"
]

print("\nUsing XGBoost model for predictions:\n")
for text in test_texts:
    result = predict_confidence(text, xgb)
    print(f"Text: '{text}'")
    print(f"Prediction: {result['confidence_label']} (score: {result['confidence_score']:.3f})")
    print()




Using XGBoost model for predictions:

Text: 'I'm absolutely certain this is the right approach.'
Prediction: high (score: 0.921)

Text: 'Um, maybe we could try that, I guess?'
Prediction: high (score: 0.634)

Text: 'I don't know, perhaps it might work.'
Prediction: low (score: 0.062)

Text: 'This is definitely the best solution.'
Prediction: high (score: 0.962)

Text: 'Maybe i will try it after this'
Prediction: high (score: 0.660)

